In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הפחתת שגיאות באמצעות רשתות טנסור (TEM): פונקציית Qiskit מאת Algorithmiq
> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן נמצאות בסטטוס שחרור תצוגה מקדימה וכפופות לשינויים.


<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## סקירה כללית
שיטת הפחתת שגיאות באמצעות רשתות טנסור (TEM) של Algorithmiq היא אלגוריתם היברידי
קוונטי-קלאסי המיועד לביצוע הפחתת רעש כולה בשלב עיבוד הנתונים הקלאסי שלאחר המדידה. עם TEM, אתה יכול לחשב
ערכי ציפייה של אופרטורים תוך הפחתת שגיאות הרעש הבלתי-נמנעות
המתרחשות בחומרה קוונטית, עם דיוק מוגבר ויעילות עלות גבוהה יותר,
מה שהופך אותה לאפשרות מושכת מאוד עבור חוקרי קוונטום ואנשי מקצוע בתעשייה כאחד.

השיטה מורכבת מבניית רשת טנסור המייצגת את ההופכי
של ערוץ הרעש הגלובלי המשפיע על מצב המעבד הקוונטי, ולאחר מכן
החלת המיפוי על תוצאות מדידה שלמות-מידעית שנרכשו מהמצב הרועש,
כדי לקבל מאמדים חסרי-הטיה עבור האופרטורים.

כיתרון, TEM מנצלת מדידות שלמות-מידעית כדי לספק
גישה למגוון רחב של ערכי ציפייה מופחתי-שגיאות של אופרטורים, ויש לה
עלות דגימה אופטימלית על החומרה הקוונטית, כמתואר ב-Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), ו-Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). עלות
המדידה מתייחסת למספר המדידות הנוספות הנדרשות לביצוע
הפחתת שגיאות יעילה, גורם קריטי בכדאיות
החישובים הקוונטיים. לפיכך, ל-TEM יש פוטנציאל לאפשר
יתרון קוונטי בתרחישים מורכבים, כגון יישומים בתחומי כאוס קוונטי,
פיזיקה של גופים רבים, דינמיקת Hubbard, וסימולציות כימיה של מולקולות קטנות.

ניתן לסכם את התכונות והיתרונות העיקריים של TEM כך:

1.  **עלות דגימה אופטימלית**: TEM היא אופטימלית ביחס
[לגבולות תיאורטיים](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
כלומר אף שיטה אינה יכולה להשיג עלות דגימה קטנה יותר. במילים אחרות,
TEM דורשת את המספר המינימלי של מדידות נוספות לביצוע
הפחתת שגיאות. משמעות הדבר היא שה-TEM משתמשת בזמן ריצה קוונטי מינימלי.
2.  **יעילות עלות**: מאחר ש-TEM מטפלת בהפחתת רעש כולה בשלב
העיבוד שלאחר המדידה, אין צורך להוסיף מעגלים נוספים למחשב הקוונטי,
מה שלא רק מוזיל את החישוב אלא גם מצמצם את
הסיכון להכנסת שגיאות נוספות עקב אי-שלמויות של
מכשירים קוונטיים.
3.  **הערכת אופרטורים מרובים**: הודות למדידות שלמות-מידעית,
TEM מעריכה ביעילות אופרטורים מרובים עם אותם
נתוני מדידה מהמחשב הקוונטי.
4.  **הפחתת שגיאות מדידה**: פונקציית ה-TEM של Qiskit כוללת גם
[שיטה קניינית להפחתת שגיאות מדידה](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
המסוגלת להפחית משמעותית שגיאות קריאה לאחר ריצת כיול קצרה.
5.  **דיוק**: TEM משפרת משמעותית את הדיוק והאמינות של
סימולציות קוונטיות דיגיטליות, מה שהופך אלגוריתמים קוונטיים למדויקים ואמינים יותר.
## תיאור
פונקציית ה-TEM מאפשרת לך לקבל ערכי ציפייה מופחתי-שגיאות עבור
אופרטורים מרובים על Circuit קוונטי עם עלות דגימה מינימלית. ה-Circuit
נמדד עם מדידת אופרטור-ערך חיובי שלמה-מידעית (IC-POVM),
ותוצאות המדידה שנאספו מעובדות במחשב
קלאסי. מדידה זו משמשת לביצוע שיטות רשת הטנסור
ובניית מיפוי היפוך-רעש. הפונקציה מחילה מיפוי שהופך
לחלוטין את ה-Circuit הרועש כולו באמצעות רשתות טנסור המייצגות
את השכבות הרועשות.

![סכמת TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "הערכה מופחתת-שגיאות של אופרטור O דרך עיבוד תוצאות מדידה לאחר המדידה מהמעבד הקוונטי הרועש. U ו-N מציינים פעולה קוונטית אידיאלית ומיפוי הרעש הנלווה, אשר יכול להיות בלתי-מקומי בדרך כלל (ולהתרחב לתיבות אפורות). D מייצג טנסור של אופרטורים שהם דואלים לאפקטים במדידת ה-IC. מודול הפחתת הרעש M הוא רשת טנסור המצומצמת ביעילות מהאמצע החוצה. האיטרציה הראשונה של הצמצום מיוצגת על ידי הקו המנוקד הסגול, השנייה על ידי הקו המקווקו, והשלישית על ידי הקו הרציף.")

לאחר שה-Circuits מוגשים לפונקציה, הם עוברים transpiling
ואופטימיזציה כדי למזער את מספר השכבות עם שערים דו-Qubit (השערים הרועשים
יותר במכשירים קוונטיים). הרעש המשפיע על השכבות נלמד דרך
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
תוך שימוש במודל רעש Pauli-Lindblad דליל כמתואר ב-E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

מודל הרעש הוא תיאור מדויק של הרעש במכשיר המסוגל
ללכוד תכונות עדינות, כולל crosstalk בין Qubits. עם זאת, הרעש על
המכשירים יכול להשתנות ולהיסחף, והרעש הנלמד עשוי שלא להיות מדויק
בנקודה שבה מתבצעת ההערכה. הדבר עלול לגרום לתוצאות לא מדויקות.
## תחילת העבודה
התחבר באמצעות [מפתח ה-API של IBM Quantum Platform](http://quantum.cloud.ibm.com/), ובחר את פונקציית ה-TEM באופן הבא. (קטע קוד זה מניח שכבר [שמרת את חשבונך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## דוגמה
קטע הקוד הבא מציג דוגמה שבה TEM משמשת לחישוב ערכי הציפייה של אופרטור עבור Circuit קוונטי פשוט.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

השתמש ב-Qiskit Serverless APIs כדי לבדוק את סטטוס עומס העבודה של פונקציית Qiskit שלך:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

אתה יכול לקבל את התוצאות כך:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** ערך הציפייה עבור ה-Circuit ללא רעש עבור האופרטור הנתון אמור להיות סביב `0.18409094298943401`.
## קלטים
**פרמטרים**

שם | סוג | תיאור | נדרש | ברירת מחדל | דוגמה
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | אוסף של אובייקטים דמויי PUB (primitive unified bloc), כגון טאפלים `(circuit, observables)` או `(circuit, observables, parameters, precision)`. ראה [סקירת PUBs](/guides/primitive-input-output#overview-of-pubs) למידע נוסף. אם מועבר Circuit שאינו ISA, הוא יעבור transpiling עם הגדרות אופטימליות. אם מועבר Circuit ISA, הוא לא יעבור transpiling; במקרה זה, האופרטור חייב להיות מוגדר על ה-QPU כולו. | כן | N/A | (circuit, observables)
`backend_name` | str | שם ה-Backend לביצוע השאילתה.| לא | אם לא מסופק, ישמש ה-Backend הפחות עמוס. | "ibm_fez"
`options` | dict | אפשרויות קלט. ראה את הסעיף `Options` לפרטים נוספים. | לא | ראה את הסעיף `Options` לפרטים נוספים.| {"max_bond_dimension": 100\}

> **Caution:** ל-TEM כרגע יש את המגבלות הבאות:
> 
>   - מעגלים עם פרמטרים אינם נתמכים. הארגומנט parameters צריך להיות מוגדר ל-`None` אם צוינה דיוק. הגבלה זו תוסר בגרסאות עתידיות.
>   - נתמכים רק מעגלים ללא לולאות. הגבלה זו תוסר בגרסאות עתידיות.
>   - שערים לא-יוניטריים, כגון reset, measure, וכל צורות ה-control flow אינם נתמכים. תמיכה ב-reset תתווסף בגרסאות הקרובות.
### אפשרויות
מילון המכיל את האפשרויות המתקדמות עבור ה-TEM. המילון עשוי להכיל את המפתחות בטבלה הבאה. אם אחת מהאפשרויות לא מסופקת, יוחל ערך ברירת המחדל המפורט בטבלה. ערכי ברירת המחדל מתאימים לשימוש רגיל ב-TEM.

שם | אפשרויות | תיאור | ברירת מחדל
-- | -- | -- | --
`tem_max_bond_dimension` | int | ממד הקשר המקסימלי לשימוש עבור רשתות הטנסור. | 500 |
`tem_compression_cutoff` | float | ערך ה-cutoff לשימוש עבור רשתות הטנסור. | 1e-16
`compute_shadows_bias_from_observable` | bool | דגל בוליאני המציין האם ה-bias לפרוטוקול מדידת הצללים הקלאסיים צריך להיות מותאם לאופרטור ה-PUB או לא. אם False, ישמש פרוטוקול הצללים הקלאסיים (הסתברות שווה למדידת Z, X, Y).| False
`shadows_bias` | np.ndarray | ה-bias לשימוש עבור פרוטוקול מדידת הצללים הקלאסיים האקראיים, מערך חד-ממדי או דו-ממדי בגודל 3 או בצורה (num_qubits, 3) בהתאמה. הסדר הוא ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int או `None` | זמן הביצוע המקסימלי על ה-QPU בשניות. אם זמן הריצה עולה על ערך זה, העבודה תבוטל. אם `None`, יוחל מגבלה ברירת מחדל שנקבעה על ידי Qiskit Runtime. | `None`
`num_randomizations` | int | מספר האקראויות לשימוש ללמידת רעש ו-gate twirling. | 32
`max_layers_to_learn` | int | המספר המקסימלי של שכבות ייחודיות ללמידה. | 4
`mitigate_readout_error` | bool | דגל בוליאני המציין האם לבצע הפחתת שגיאות קריאה או לא. | True
`num_readout_calibration_shots` | int | מספר הירויות לשימוש להפחתת שגיאות קריאה. | 10000
`default_precision` | float | הדיוק ברירת המחדל לשימוש עבור ה-PUBs שלא צוין עבורם דיוק. |0.02
`seed` | int או `None` | הגדר את ה-seed של מחולל המספרים האקראיים לצורך שחזוריות. אם `None`, אל תגדיר את ה-seed. | None
## פלטים
[PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) של Qiskit המכיל את התוצאה המופחתת-שגיאות על ידי TEM. התוצאה עבור כל PUB מוחזרת כ-[PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) המכיל את השדות הבאים:

שם | סוג | תיאור
-- | -- | --
data | DataBin | [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) של Qiskit המכיל את האופרטור המופחת-שגיאות על ידי TEM ושגיאת התקן שלו. ל-DataBin יש את השדות הבאים: <ul><li>`evs`: ערך האופרטור המופחת-שגיאות על ידי TEM.</li><li>`stds`: שגיאת התקן של האופרטור המופחת-שגיאות על ידי TEM.</li></ul>
metadata | dict | מילון המכיל תוצאות נוספות. המילון מכיל את המפתחות הבאים: <ul><li>`"evs_non_mitigated"`: ערך האופרטור ללא הפחתת שגיאות.</li><li>`"stds_non_mitigated"`: שגיאת התקן של התוצאה ללא הפחתת שגיאות.</li><li>`"evs_mitigated_no_readout_mitigation"`: ערך האופרטור עם הפחתת שגיאות אך ללא הפחתת שגיאות קריאה.</li><li>`"stds_mitigated_no_readout_mitigation"`: שגיאת התקן של התוצאה עם הפחתת שגיאות אך ללא הפחתת שגיאות קריאה.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: ערך האופרטור ללא הפחתת שגיאות אך עם הפחתת שגיאות קריאה.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: שגיאת התקן של התוצאה ללא הפחתת שגיאות אך עם הפחתת שגיאות קריאה.</li></ul>
## אחזור הודעות שגיאה
אם סטטוס העומס שלך הוא ERROR, השתמש ב-job.result() כדי לאחזר את הודעת השגיאה באופן הבא: